# ATLAS — Unified NAIP-targeted USGS DEM acquisition (180 facilities)

**Objective:** Standardize **all 180** DEMs using one reproducible raster-grid specification derived from reconstructed NAIP geographic footprints. This supersedes the missing-only 102/78 split. **Do not assume the reconstructed NAIP footprints are verified historical image boundaries.**

**Safety:** All new outputs are local under `data/temp/usgs_naip_aligned_v2/`; publication is opt-in to a new `usgs/naip_aligned_v2/` S3 prefix. Existing `usgs/raw/` and historical manifests are never modified by this notebook. The superseded v1 S3 collection was retired separately. Run top to bottom. Requires `boto3`, `rasterio`, `shapely`, `requests`, `pandas`, `numpy`, `pyarrow`.

## 1 — Inventory current NAIP images and existing DEMs

Set the S3 prefixes and list NAIP PNGs and DEM GeoTIFFs already in the bucket. This is the inventory the footprint and manifest cells build on.


In [ ]:
from pathlib import Path
import json
import re
import boto3
import numpy as np
import pandas as pd
import rasterio
from rasterio.session import AWSSession
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds, Window

BUCKET = "atlas-acm-research-f26"
SATELLITE_PREFIX = "satellite/raw/"
USGS_PREFIX = "usgs/raw/"
METADATA_PREFIX = "satellite/metadata/"

cwd = Path.cwd().resolve()
repo_root = next((p for p in (cwd, *cwd.parents)
                  if (p / "data/processed/im3_facility_registry.csv").exists()), None)
if repo_root is None:
    raise FileNotFoundError("Cannot locate data/processed/im3_facility_registry.csv")
OUTPUT = repo_root / "data/processed/usgs_naip_alignment"
OUTPUT.mkdir(parents=True, exist_ok=True)
s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")
print("Repository:", repo_root)

def inventory(prefix, suffix):
    records = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            name = Path(key).name
            if not name.lower().endswith(suffix):
                continue
            fid = name[:-len(suffix)]
            if not re.fullmatch(r"\d+", fid):
                continue
            records.append({"id": fid.zfill(10), "s3_key": key})
    result = pd.DataFrame(records, columns=["id", "s3_key"])
    assert result["id"].is_unique, f"Duplicate facility IDs in {prefix}"
    return result

sat = inventory(SATELLITE_PREFIX, ".png").rename(columns={"s3_key": "naip_s3_key"})
dems = inventory(USGS_PREFIX, "_dem.tif").rename(columns={"s3_key": "usgs_s3_key"})
satellite_ids = set(sat["id"])
usgs_keys = dict(zip(dems["id"], dems["usgs_s3_key"]))
print("Satellite:", len(sat), "| Existing DEMs:", len(dems),
      "| Matched:", len(satellite_ids & set(usgs_keys)))

**Read reconstructed NAIP footprints**

Load each facility JSON under the satellite metadata prefix and turn its polygon into a longitude/latitude bounding box. These boxes are reconstructed metadata, not a separately verified historical image extent.


In [ ]:
metadata_keys = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=METADATA_PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        # Only facility JSONs directly under satellite/metadata/.
        if (Path(key).parent.as_posix() == METADATA_PREFIX.rstrip("/")
                and re.fullmatch(r"\d{10}\.json", Path(key).name)):
            metadata_keys.append(key)

records = []
for key in sorted(metadata_keys):
    raw = json.loads(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read())
    fid = str(raw["id"]).strip().zfill(10)
    assert Path(key).stem == fid, f"ID/file mismatch: {key}"
    region = raw["region_geojson"]
    region = json.loads(region) if isinstance(region, str) else region
    assert region["type"] == "Polygon", f"Unexpected geometry: {key}"
    ring = region["coordinates"][0]
    lons, lats = zip(*[(float(pt[0]), float(pt[1])) for pt in ring])
    records.append({
        "id": fid, "metadata_key": key, "metadata_naip_key": raw.get("s3_key"),
        "metadata_status": raw.get("status"),
        "min_lon": min(lons), "min_lat": min(lats),
        "max_lon": max(lons), "max_lat": max(lats),
        "naip_year": raw.get("year"), "buffer_meters": raw.get("buffer_meters"),
        "metadata_provenance": raw.get("metadata_provenance")
    })
metadata_df = pd.DataFrame(records)
assert metadata_df["id"].is_unique
assert metadata_df[["min_lon", "min_lat", "max_lon", "max_lat"]].notna().all().all()
assert ((metadata_df.min_lon < metadata_df.max_lon) &
        (metadata_df.min_lat < metadata_df.max_lat)).all()
print("Parsed facility metadata:", len(metadata_df))
print("Satellite missing metadata:", len(satellite_ids - set(metadata_df.id)))
print("Metadata missing satellite:", len(set(metadata_df.id) - satellite_ids))
assert satellite_ids == set(metadata_df.id), "Investigate metadata/satellite mismatch"
display(metadata_df.head())

**Join imagery, footprints, and the facility registry**

Merge the NAIP keys, those boxes, and registry coordinates into one 180-row manifest. The cell stops if an id is duplicated or a coordinate is missing.


In [ ]:
registry = pd.read_csv(repo_root / "data/processed/im3_facility_registry.csv", dtype={"id": str})
registry["id"] = registry["id"].str.strip().str.zfill(10)
assert registry.id.is_unique
manifest = (sat.merge(metadata_df, on="id", validate="one_to_one")
            .merge(registry[["id", "lat", "lon", "name", "state"]], on="id", how="left", validate="one_to_one")
            .sort_values("id").reset_index(drop=True))
assert len(manifest) == 180 and manifest.id.nunique() == 180
assert manifest[["lat", "lon"]].notna().all().all()
assert (manifest.naip_s3_key == manifest.metadata_naip_key).all()
print("Unified target facilities:", len(manifest))
display(manifest[["id", "naip_s3_key", "metadata_key", "min_lon", "min_lat", "max_lon", "max_lat"]].head())

## 2 — Fixed output grid and safe paths

Set the shared output grid: WGS84, one float32 elevation band, and a 1/10800-degree cell size anchored to global coordinates. Also define the local output folders and the helpers that snap a facility box onto that grid.


In [ ]:
import os, time, hashlib, requests
from datetime import datetime, timezone
from contextlib import ExitStack
from urllib.parse import urlparse
from shapely.geometry import box
from shapely.ops import unary_union
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin, array_bounds
from botocore.exceptions import ClientError

USGS_API = "https://tnmaccess.nationalmap.gov/api/v1/products"
USGS_DATASET = "National Elevation Dataset (NED) 1/3 arc-second"
RASTER_HTTP_OPTIONS = {"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR", "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif", "GDAL_HTTP_TIMEOUT": "60"}
TARGET_CRS = "EPSG:4326"
TARGET_RES_DEG = 1 / 10800
NODATA = -9999.0
VERSION = "v2"
ALIGNED_DEM_DIR = repo_root / "data/temp/usgs_naip_aligned_v2"
CHECKPOINT_DIR = Path.home() / "atlas_usgs_checkpoints"  # Avoid OneDrive locking
CHECKPOINT = CHECKPOINT_DIR / "usgs_naip_v2_checkpoint.csv"
PUBLISH_PREFIX = "usgs/naip_aligned_v2"
MANIFEST_LOCAL = OUTPUT / "usgs_naip_alignment_manifest_v2.parquet"

def target_bbox(row):
    b = tuple(float(row[k]) for k in ("min_lon", "min_lat", "max_lon", "max_lat"))
    if not (-180 <= b[0] < b[2] <= 180 and -90 <= b[1] < b[3] <= 90):
        raise ValueError(f"Invalid target bounds: {b}")
    return b

def grid_for_bbox(bbox):
    # Snap outward to a common globally anchored EPSG:4326 grid.
    r = TARGET_RES_DEG
    west = np.floor(bbox[0] / r) * r
    east = np.ceil(bbox[2] / r) * r
    south = np.floor(bbox[1] / r) * r
    north = np.ceil(bbox[3] / r) * r
    width = int(round((east - west) / r))
    height = int(round((north - south) / r))
    assert width > 0 and height > 0
    return from_origin(west, north, r, r), width, height

print("Targets:", len(manifest), "| output:", ALIGNED_DEM_DIR, "| S3:", PUBLISH_PREFIX)

## 3 — USGS source discovery and selection

Define helpers that page the National Map for 1/3 arc-second tiles and choose either one tile that covers the target box or a same-resolution mosaic that does.


In [ ]:
def discover_dem_urls(bbox, http=None):
    http = http or requests.Session()
    params = {"bbox": ",".join(map(str, bbox)), "datasets": USGS_DATASET,
              "max": 100, "offset": 0}
    urls = []
    for _ in range(30):
        for attempt in range(3):
            try:
                response = http.get(USGS_API, params=params, timeout=60)
                response.raise_for_status()
                payload = response.json()
                break
            except (requests.RequestException, ValueError):
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)
        items = payload.get("items", [])
        if not isinstance(items, list):
            raise ValueError("Unexpected USGS API items payload")
        urls.extend(item["downloadURL"] for item in items
                    if isinstance(item.get("downloadURL"), str)
                    and urlparse(item["downloadURL"]).path.lower().endswith((".tif", ".tiff")))
        params["offset"] += len(items)
        total = payload.get("total")
        if total is not None and params["offset"] >= int(total):
            break
        if not items:
            break
        if total is None and len(items) < params["max"]:
            break
    else:
        raise RuntimeError("USGS pagination limit reached")
    return list(dict.fromkeys(urls))


def inspect_dem_urls(bbox, urls):
    inspected = []
    with rasterio.Env(**RASTER_HTTP_OPTIONS):
        for url in urls:
            try:
                with rasterio.open(url) as src:
                    if src.count != 1 or src.crs is None or src.nodata is None:
                        raise ValueError("Expected single-band georeferenced DEM with nodata")
                    target = transform_bounds("EPSG:4326", src.crs, *bbox, densify_pts=21)
                    footprint = box(*src.bounds)
                    filename = Path(urlparse(url).path).name
                    match = re.search(r"USGS_13_(n\d{2}w\d{3})_(\d{8})", filename, re.I)
                    inspected.append({"url": url, "filename": filename,
                                      "tile_id": match.group(1) if match else filename,
                                      "date": match.group(2) if match else "00000000",
                                      "crs": str(src.crs), "res": tuple(src.res),
                                      "nodata": src.nodata, "bounds": tuple(src.bounds),
                                      "covers": footprint.covers(box(*target)),
                                      "intersects": footprint.intersects(box(*target)),
                                      "status": "ok"})
            except Exception as exc:
                inspected.append({"url": url, "status": "error", "error": str(exc)})
    return pd.DataFrame(inspected)


def select_dem_sources(bbox, inspected):
    if inspected.empty or "status" not in inspected:
        raise ValueError("No USGS DEM candidates")
    ok = inspected.loc[inspected.status.eq("ok")].copy()
    if ok.empty:
        raise ValueError("No readable USGS DEM candidates")
    ok = ok.sort_values(["date", "filename"], ascending=[False, True])
    singles = ok.loc[ok.covers]
    if not singles.empty:
        return "single", singles.head(1)
    selected = ok.loc[ok.intersects].drop_duplicates("tile_id", keep="first")
    if selected.empty:
        raise ValueError("No intersecting DEM sources")
    if selected.crs.nunique() != 1:
        raise ValueError("Mixed source CRS")
    first = selected.iloc[0]
    for row in selected.itertuples(index=False):
        if not np.allclose(row.res, first.res, rtol=1e-6, atol=0):
            raise ValueError("Mixed DEM resolutions")
        if row.nodata != first.nodata:
            raise ValueError("Mixed DEM nodata conventions")
    target = box(*transform_bounds("EPSG:4326", first.crs, *bbox, densify_pts=21))
    if not unary_union([box(*b) for b in selected.bounds]).covers(target):
        raise ValueError("Selected sources do not fully cover NAIP bbox")
    return "mosaic", selected


## 4 — Uniform extraction and strict QA

Define the acquisition used for all 180 facilities: reuse an existing DEM when it already fills the target grid, otherwise reproject or fetch USGS tiles until every target pixel is valid. A reprojected file keeps separate source and resampling provenance.


In [ ]:
def head_or_none(key):
    try:
        return s3.head_object(Bucket=BUCKET, Key=key)
    except ClientError as exc:
        if exc.response["Error"]["Code"] in ("404", "NoSuchKey"):
            return None
        raise

def validate_v2(path, row, provenance_path=None):
    bbox = target_bbox(row)
    expected_transform, expected_width, expected_height = grid_for_bbox(bbox)
    with rasterio.open(path) as src:
        assert src.count == 1 and str(src.crs) == TARGET_CRS
        assert src.width == expected_width and src.height == expected_height
        assert np.allclose(tuple(src.transform)[:6], tuple(expected_transform)[:6], atol=1e-9, rtol=0)
        assert src.dtypes[0] == "float32"
        a = src.read(1, masked=True)
        valid = float(np.ma.count(a) / a.size)
        assert valid == 1.0 and np.isfinite(a.compressed()).all(), f"Invalid elevation pixels: {valid}"
        bounds = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
        tol = 1e-9
        assert bounds[0] <= bbox[0]+tol and bounds[1] <= bbox[1]+tol
        assert bounds[2] >= bbox[2]-tol and bounds[3] >= bbox[3]-tol
        quality = {"valid_fraction": valid, "shape": [src.height, src.width],
                   "crs": str(src.crs), "resolution": list(src.res),
                   "grid_transform": list(src.transform)[:6], "raster_bounds_wgs84": list(bounds)}
    if provenance_path is not None:
        record = json.loads(Path(provenance_path).read_text())
        assert record["id"] == str(row["id"]).zfill(10)
        assert np.allclose(record["target_bbox_wgs84"], bbox, atol=1e-10, rtol=0)
        assert record["naip_metadata_key"] == row["metadata_key"]
        assert record["processing_version"] == VERSION
        assert record["target_crs"] == TARGET_CRS
        assert record["target_resolution_degrees"] == TARGET_RES_DEG
        assert record["footprint_verified_historical_export"] is False
        assert record.get("source_urls") or record.get("input_dem_s3_key")
    return quality


def acquire_v2(row):
    fid = str(row["id"]).zfill(10)
    bbox = target_bbox(row)
    ALIGNED_DEM_DIR.mkdir(parents=True, exist_ok=True)
    tif = ALIGNED_DEM_DIR / f"{fid}_dem.tif"
    js = ALIGNED_DEM_DIR / f"{fid}_dem.json"
    if tif.exists() or js.exists():
        if not (tif.exists() and js.exists()):
            raise FileExistsError(f"Incomplete output for {fid}; inspect before retry")
        return {"id": fid, "status": "skipped_valid", **validate_v2(tif, row, js)}

    transform, width, height = grid_for_bbox(bbox)
    target = np.full((height, width), NODATA, dtype="float32")
    candidates = [
        f"usgs/raw/{fid}_dem.tif",
    ]
    reuse_key = None
    reuse_errors = []
    for key in candidates:
        if head_or_none(key) is None:
            continue
        # A single candidate is downloaded temporarily, not the entire dataset.
        cached = ALIGNED_DEM_DIR / f"{fid}_candidate.partial.tif"
        try:
            s3.download_file(BUCKET, key, str(cached))
            target.fill(NODATA)
            with rasterio.open(cached) as src:
                if src.count != 1 or src.crs is None:
                    raise ValueError("Candidate is not a single-band georeferenced raster")
                reproject(source=rasterio.band(src, 1), destination=target,
                          src_transform=src.transform, src_crs=src.crs,
                          src_nodata=src.nodata, dst_transform=transform,
                          dst_crs=TARGET_CRS, dst_nodata=NODATA,
                          resampling=Resampling.bilinear)
            if np.all(np.isfinite(target) & (target != NODATA)):
                reuse_key = key
                break
            reuse_errors.append(f"{key}: incomplete target-grid coverage")
        except Exception as exc:
            reuse_errors.append(f"{key}: {type(exc).__name__}: {exc}")
        finally:
            cached.unlink(missing_ok=True)

    if reuse_key is not None:
        method = "reproject_existing_s3_dem"
        source_urls = []  # Original source tile URLs are not inferred from a cached DEM.
        source_filenames = []
    else:
        # Only call USGS when neither existing S3 raster covers the new grid.
        urls = discover_dem_urls(bbox)
        inspected = inspect_dem_urls(bbox, urls)
        method, chosen = select_dem_sources(bbox, inspected)
        with rasterio.Env(**RASTER_HTTP_OPTIONS):
            with ExitStack() as stack:
                sources = [stack.enter_context(rasterio.open(url)) for url in chosen.url]
                reference = sources[0]
                crop_bounds = transform_bounds(TARGET_CRS, reference.crs, *bbox, densify_pts=21)
                pad_x, pad_y = (abs(reference.res[0]) * 3, abs(reference.res[1]) * 3)
                padded = (crop_bounds[0]-pad_x, crop_bounds[1]-pad_y,
                          crop_bounds[2]+pad_x, crop_bounds[3]+pad_y)
                mosaic, src_transform = merge(sources, bounds=padded, res=reference.res,
                                              nodata=reference.nodata, method="first")
                target.fill(NODATA)
                reproject(source=mosaic[0], destination=target,
                          src_transform=src_transform, src_crs=reference.crs,
                          src_nodata=reference.nodata, dst_transform=transform,
                          dst_crs=TARGET_CRS, dst_nodata=NODATA,
                          resampling=Resampling.bilinear)
        source_urls = chosen.url.tolist()
        source_filenames = chosen.filename.tolist()
    tmp_tif = ALIGNED_DEM_DIR / f"{fid}_dem.partial.tif"
    tmp_json = ALIGNED_DEM_DIR / f"{fid}_dem.partial.json"
    if tmp_tif.exists() or tmp_json.exists():
        raise FileExistsError(f"Stale partial files for {fid}")
    profile = {"driver": "GTiff", "height": height, "width": width,
               "count": 1, "dtype": "float32", "crs": TARGET_CRS,
               "transform": transform, "nodata": NODATA, "compress": "deflate"}
    provenance = {"id": fid, "processing_version": VERSION,
        "processing_method": "USGS source extraction and bilinear reprojection to globally anchored EPSG:4326 grid",
        "target_bbox_wgs84": list(bbox), "target_crs": TARGET_CRS,
        "target_resolution_degrees": TARGET_RES_DEG,
        "naip_s3_key": row["naip_s3_key"], "naip_metadata_key": row["metadata_key"],
        "naip_metadata_provenance": row["metadata_provenance"],
        "footprint_verified_historical_export": False,
        "usgs_dataset": USGS_DATASET, "selection_method": method,
        "source_urls": source_urls, "source_filenames": source_filenames,
        "input_dem_s3_key": reuse_key, "reuse_attempt_notes": reuse_errors,
        "input_provenance_status": "existing_source_provenance_not_independently_reverified" if reuse_key else None,
        "created_utc": datetime.now(timezone.utc).isoformat()}
    try:
        with rasterio.open(tmp_tif, "w", **profile) as dst:
            dst.write(target, 1)
        quality = validate_v2(tmp_tif, row)
        provenance.update(quality)
        tmp_json.write_text(json.dumps(provenance, indent=2))
        # Do not overwrite existing files, including on rerun.
        os.link(tmp_tif, tif)
        try:
            os.link(tmp_json, js)
        except Exception:
            tif.unlink(missing_ok=True)
            raise
        return {"id": fid, "status": "success", **quality}
    finally:
        tmp_tif.unlink(missing_ok=True)
        tmp_json.unlink(missing_ok=True)

## 5 — Pilot

Run acquisition for the first manifest facility when the pilot flag is on. Leave it off to print the target box without downloading.


In [ ]:
RUN_PILOT = False
pilot = manifest.iloc[0]
print("Pilot:", pilot.id, "bbox:", target_bbox(pilot))
if RUN_PILOT:
    print(acquire_v2(pilot))
else:
    print("Pilot disabled. Set RUN_PILOT=True to acquire one facility.")

**Resumable batch**

Define the batch runner. With execution off it only prints the target rows; with execution on it acquires each facility and writes a checkpoint so a stopped run can continue.


In [ ]:
def run_v2_batch(rows, execute=False):
    rows = rows.copy()
    assert rows.id.is_unique
    assert set(rows.id).issubset(set(manifest.id))
    if not execute:
        print(f"DRY RUN: {len(rows)} targets; no downloads or writes")
        display(rows[["id", "min_lon", "min_lat", "max_lon", "max_lat"]].head())
        return None
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    previous = pd.read_csv(CHECKPOINT, dtype={"id": str}) if CHECKPOINT.exists() else pd.DataFrame()
    latest = {r["id"]: r for r in previous.to_dict("records")} if not previous.empty else {}
    for _, row in rows.iterrows():
        try:
            result = acquire_v2(row)
        except Exception as exc:
            result = {"id": row.id, "status": "error", "error": f"{type(exc).__name__}: {exc}"}
        latest[row.id] = result
        print(row.id, result["status"], result.get("error", ""))
        tmp = CHECKPOINT.with_suffix(".partial.csv")
        pd.DataFrame(latest.values()).to_csv(tmp, index=False)
        os.replace(tmp, CHECKPOINT)
    return pd.DataFrame(latest.values())

RUN_ALL_180 = False  # Opt in only after successful pilot
batch_results = run_v2_batch(manifest, execute=RUN_ALL_180)
if batch_results is not None:
    print(batch_results.status.value_counts(dropna=False))

## 6 — Full local QA gate and manifest construction

Open every local v2 DEM and provenance file, run the strict grid check, and build the aligned manifest only if all 180 pass.


In [ ]:
qa_rows = []
for _, row in manifest.iterrows():
    fid = row.id
    tif = ALIGNED_DEM_DIR / f"{fid}_dem.tif"
    js = ALIGNED_DEM_DIR / f"{fid}_dem.json"
    if not (tif.exists() and js.exists()):
        qa_rows.append({"id": fid, "passed": False, "error": "Missing DEM or provenance"})
        continue
    try:
        metrics = validate_v2(tif, row, js)
        qa_rows.append({"id": fid, "passed": True, **metrics})
    except Exception as exc:
        qa_rows.append({"id": fid, "passed": False, "error": f"{type(exc).__name__}: {exc}"})
qa_df = pd.DataFrame(qa_rows)
print("QA passed:", int(qa_df.passed.sum()), "/", len(qa_df))
display(qa_df.loc[~qa_df.passed])

if qa_df.passed.all() and len(qa_df) == 180:
    aligned_manifest = manifest[["id", "naip_s3_key", "metadata_key", "metadata_provenance", "lat", "lon"]].copy()
    aligned_manifest["usgs_s3_key"] = PUBLISH_PREFIX + "/dem/" + aligned_manifest.id + "_dem.tif"
    aligned_manifest["provenance_s3_key"] = PUBLISH_PREFIX + "/provenance/" + aligned_manifest.id + "_dem.json"
    aligned_manifest["processing_version"] = VERSION
    aligned_manifest["provenance_status"] = "see_per_facility_provenance"
    aligned_manifest["alignment_qa_status"] = "target_grid_and_valid_pixels_verified"
    aligned_manifest["naip_footprint_verified"] = False
    aligned_manifest = aligned_manifest.merge(qa_df.drop(columns=["error"], errors="ignore"), on="id", validate="one_to_one")
    assert aligned_manifest.id.nunique() == 180
    aligned_manifest.to_parquet(MANIFEST_LOCAL, index=False)
    print("Manifest written:", MANIFEST_LOCAL)
else:
    print("Publication BLOCKED until all 180 pass QA")

## 7 — Publication list

Pair each local DEM and provenance file with its `usgs/naip_aligned_v2/` key, and report whether the QA gate passed and whether any destination key already exists. This cell does not upload.


In [ ]:
publication_items = []
for fid in manifest.id:
    publication_items += [
        (ALIGNED_DEM_DIR / f"{fid}_dem.tif", f"{PUBLISH_PREFIX}/dem/{fid}_dem.tif"),
        (ALIGNED_DEM_DIR / f"{fid}_dem.json", f"{PUBLISH_PREFIX}/provenance/{fid}_dem.json")]
ready = qa_df.passed.all() and len(qa_df) == 180 and MANIFEST_LOCAL.exists()
print("QA gate:", ready, "| target objects:", len(publication_items))
if ready:
    missing_local = [(str(p), k) for p, k in publication_items if not p.exists()]
    collisions = [k for p, k in publication_items if head_or_none(k) is not None]
    print("Missing local:", len(missing_local), "| destination collisions:", len(collisions))
    assert not missing_local and not collisions
else:
    print("Not ready: complete acquisition and QA first")

**Publish only when opted in**

Upload those objects when the publish flag is on, and refuse to overwrite a key that already exists. The manifest is sent only after the DEM and provenance objects are in place.


In [ ]:
PUBLISH_V2 = False  # Never set True before the preflight succeeds
if PUBLISH_V2:
    assert ready and not missing_local and not collisions
    for path, key in publication_items:
        if head_or_none(key) is not None:
            raise FileExistsError(f"Refusing overwrite: {key}")
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        s3.upload_file(str(path), BUCKET, key, ExtraArgs={"Metadata": {"sha256": digest}})
        remote = head_or_none(key)
        assert remote["ContentLength"] == path.stat().st_size
        assert remote["Metadata"].get("sha256") == digest
    print("Verified 360 DEM/provenance objects")
    # Publish the discoverable manifest only after all files exist.
    manifest_key = "usgs/processed/usgs_naip_alignment_manifest_v2.parquet"
    assert head_or_none(manifest_key) is None, "Refusing manifest overwrite"
    s3.upload_file(str(MANIFEST_LOCAL), BUCKET, manifest_key)
    assert head_or_none(manifest_key)["ContentLength"] == MANIFEST_LOCAL.stat().st_size
    print("Published manifest:", f"s3://{BUCKET}/{manifest_key}")
else:
    print("Publication disabled")

## Research limitations and handoff

- The 180 target footprints are **reconstructed** from available NAIP metadata, not independently verified historical PNG export extents.
- All v2 rasters follow one geographic grid rule, but this **does not establish pixel-level NAIP/DEM co-registration**. NAIP imagery should be georeferenced from verified export metadata before claiming that.
- The elevation dataset and NAIP acquisition dates may differ; preserve source URLs and dates.
- Bilinear reprojection smooths elevation values. Terrain CNN preprocessing should use appropriate elevation normalization and derived features, with train/test splitting by facility or geography to avoid leakage.
- Original 262-file historical collection and `v1` remain unchanged; only switch downstream training to `v2` after full QA and S3 publication.